In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
import tensorflow as tf
import numpy as np


2023-08-14 20:22:23.561021: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-08-14 20:22:24.881630: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
input_file = 'input.wav'
output_file = 'rangemaster.wav'

input_wav = tf.io.read_file(input_file)
output_wav = tf.io.read_file(output_file)

2023-08-14 20:22:26.805932: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
channels = 1

input_lookback = 441

In [5]:
input_samples, sample_rate = tf.audio.decode_wav(input_wav)
output_samples, sample_rate = tf.audio.decode_wav(output_wav)
sample_len = 44100 * 10 #min(len(input_samples), len(output_samples)) 

In [6]:
# input_data = np.zeros((sample_len, input_lookback, 1))
output_data = np.zeros((sample_len, 1))

skip = 44100 * 30

for i in range(sample_len):
  #     input_data[i] = input_samples[skip + i : skip + i + input_lookback]
  output_data[i] = output_samples[skip + i + input_lookback - 1]

# np.save('input.npy', input_data)
np.save(output_file + '.npy', output_data)

input_data = np.load('input.npy')
# output_data = np.load(output_file + '.npy') 

In [7]:
model = tf.keras.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dense(1)
])

In [8]:
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

In [9]:
history = model.fit(
    x = input_data,
    y = output_data,
    validation_split = 0.1,
    shuffle = True,
    epochs=5)



Epoch 1/5
12404/12404 [==============================] - 26s 2ms/step - loss: 7.4276e-04 - mae: 0.0178 - val_loss: 0.0017 - val_mae: 0.0295
Epoch 2/5
12404/12404 [==============================] - 25s 2ms/step - loss: 6.1338e-04 - mae: 0.0163 - val_loss: 0.0016 - val_mae: 0.0274
Epoch 3/5
12404/12404 [==============================] - 25s 2ms/step - loss: 5.7123e-04 - mae: 0.0157 - val_loss: 0.0015 - val_mae: 0.0276
Epoch 4/5
12404/12404 [==============================] - 25s 2ms/step - loss: 5.4741e-04 - mae: 0.0154 - val_loss: 0.0014 - val_mae: 0.0264
Epoch 5/5
12404/12404 [==============================] - 25s 2ms/step - loss: 5.3101e-04 - mae: 0.0151 - val_loss: 0.0014 - val_mae: 0.0250


In [10]:
model.summary()

prediction = model.predict(input_data)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 441)               0         
                                                                 
 dense (Dense)               (None, 16)                7072      
                                                                 
 dense_1 (Dense)             (None, 16)                272       
                                                                 
 dense_2 (Dense)             (None, 16)                272       
                                                                 
 dense_3 (Dense)             (None, 1)                 17        
                                                                 
Total params: 7,633
Trainable params: 7,633
Non-trainable params: 0
_________________________________________________________________
13782/13782 [==============================] - 19s 1ms

In [ ]:
prediction_audio = tf.audio.encode_wav(prediction, 44100)
tf.io.write_file('prediction.wav', prediction_audio)